# Notebook 03 - LiDAR Extraction & Temporal Alignment

**Project:** ECE 5424 Advanced ML Capstone - SNN vs CNN for LiDAR-Event Camera Fusion
**Sequence:** `zurich_city_04_a` (primary)  
**Goal:** Extract LiDAR point clouds from the ROS1 bag, verify temporal overlap with the event camera, and produce a JSON manifest pairing each semantic segmentation frame with its nearest LiDAR sweep.

**Steps:**
1. Environment setup & bag inspection  
2. Extract LiDAR timestamps → verify temporal overlap with events  
3. Extract point clouds for the overlap window → save as `.npy`  
4. Temporal matching: pair each semantic frame with its nearest LiDAR sweep  

>  **DO NOT** commit `lidar_sweeps/` or `*.npy` files to git,  already covered by `.gitignore`.
>
> **DO NOT** process `zurich_city_00_a` here. Session 2 handles projection.

#### Why Temporal Alignment Is the Core Problem in Sensor Fusion

Before writing any code, it helps to understand what problem we are actually solving.

The event camera and the LiDAR are two independent sensors recording the same scene, but they do not share a clock. The event camera stores timestamps relative to when it started recording (its own stopwatch starting at zero). The LiDAR stores absolute ROS timestamps that had been running for hours before the event camera turned on.

If you try to match them directly, every event timestamp appears to be roughly 10 hours before every LiDAR sweep. The overlap would be zero and you would get zero training pairs.

The fix is `t_offset`: a scalar stored in `events.h5` that tells you how many microseconds the LiDAR clock had already counted before the event camera started. Adding this offset to every event timestamp puts both sensors on the same absolute timeline.

A second subtlety: even inside the overlap window, both sensors are never writing at the exact same microsecond. The LiDAR fires at roughly 10 Hz (one sweep every 100ms). The semantic label frames also run at 10 Hz. But they started independently, so each label frame lands at a slightly different moment than its nearest LiDAR sweep, on average about 25ms apart. This matters because a car traveling at 50 km/h moves roughly 35cm in 25ms, and almost 70cm in 50ms. A training sample with a 70cm positional mismatch between the label and the depth map is teaching the model wrong geometry.

This is why we apply a 55ms quality filter: any frame/sweep pair further apart than half a LiDAR sweep period gets rejected, because at that point the scene has changed too much for the pair to be a reliable training sample.

## Paths:  load from config.py

All local paths are derived from `DATA_ROOT` in `config.py`
Copy `config.example.py` → `config.py` and fill in your local path before running.

In [1]:
import sys
from pathlib import Path

# Repo root is one level up from notebooks/
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from config import DATA_ROOT

DATA_ROOT = Path(DATA_ROOT)
SEQUENCE  = "zurich_city_04_a"
SEQ_DIR   = DATA_ROOT / SEQUENCE

#  All paths derived from DATA_ROOT
BAG_PATH        = SEQ_DIR / "lidar_imu.bag"
EVENTS_PATH     = SEQ_DIR / f"{SEQUENCE}_events_left" / "events.h5"
TIMESTAMPS_PATH = SEQ_DIR / f"{SEQUENCE}_disparity_timestamps.txt"
BOUNDS_PATH     = SEQ_DIR / "overlap_bounds_04a.npy"
OUTPUT_DIR      = SEQ_DIR / "lidar_sweeps"
PAIRS_OUTPUT    = REPO_ROOT / "data"
#Uncomment this print to check if everything is okay
"""print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"DATA_ROOT   : {DATA_ROOT}")
print(f"SEQ_DIR     : {SEQ_DIR}")
print()
for label, p in [
    ("BAG_PATH       ", BAG_PATH),
    ("EVENTS_PATH    ", EVENTS_PATH),
    ("TIMESTAMPS_PATH", TIMESTAMPS_PATH),
    ("OUTPUT_DIR     ", OUTPUT_DIR),
    ("PAIRS_OUTPUT   ", PAIRS_OUTPUT),
]:
    exists = "✓" if p.exists() else "✗ NOT FOUND"
    print(f"  {label}: {p}  [{exists}]")"""

'print(f"REPO_ROOT   : {REPO_ROOT}")\nprint(f"DATA_ROOT   : {DATA_ROOT}")\nprint(f"SEQ_DIR     : {SEQ_DIR}")\nprint()\nfor label, p in [\n    ("BAG_PATH       ", BAG_PATH),\n    ("EVENTS_PATH    ", EVENTS_PATH),\n    ("TIMESTAMPS_PATH", TIMESTAMPS_PATH),\n    ("OUTPUT_DIR     ", OUTPUT_DIR),\n    ("PAIRS_OUTPUT   ", PAIRS_OUTPUT),\n]:\n    exists = "✓" if p.exists() else "✗ NOT FOUND"\n    print(f"  {label}: {p}  [{exists}]")'

## 1. Environment Setup & Bag Inspection

Install dependencies if not already present, then inspect the bag topics without extracting any data.

In [19]:
# Install / upgrade required packages
import subprocess

pkgs = ["rosbags", "numpy", "h5py", "hdf5plugin", "pyyaml", "matplotlib", "opencv-python"]

for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", pkg],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗"
    print(f"  {status} {pkg}")

print("\nDependency check complete.")

  ✓ rosbags
  ✓ numpy
  ✓ h5py
  ✓ hdf5plugin
  ✓ pyyaml
  ✓ matplotlib
  ✓ opencv-python

Dependency check complete.


In [20]:
# Bag inspection
from rosbags.rosbag1 import Reader

with Reader(BAG_PATH) as reader:
    print("Topics in bag:")
    for topic, conn in reader.topics.items():
        print(f"  {topic:50s}  {conn.msgtype}")

    duration_s = reader.duration / 1e9
    print(f"\nDuration      : {duration_s:.1f} seconds")
    print(f"Message count : {reader.message_count:,}")

Topics in bag:
  /imu/data                                           sensor_msgs/msg/Imu
  /velodyne_points                                    sensor_msgs/msg/PointCloud2

Duration      : 630.2 seconds
Message count : 634,582


In [22]:
# Identify topic names for later steps
with Reader(BAG_PATH) as reader:
    lidar_topics = [
        t for t in reader.topics
        if any(kw in t.lower() for kw in ["lidar", "velodyne", "points"])
    ]
    imu_topics = [
        t for t in reader.topics
        if "imu" in t.lower()
    ]

print("LiDAR topic(s) detected:", lidar_topics)
print("IMU topic(s) detected   :", imu_topics)

assert len(lidar_topics) >= 1, "No LiDAR topic found — check keyword list above!"
LIDAR_TOPIC = lidar_topics[0]
print(f"\n Using LIDAR_TOPIC = '{LIDAR_TOPIC}'")

LiDAR topic(s) detected: ['/velodyne_points']
IMU topic(s) detected   : ['/imu/data']

 Using LIDAR_TOPIC = '/velodyne_points'



## 2.Extract LiDAR Timestamps & Verify Temporal Overlap with Events

Read only timestamps from the bag (fast, no point cloud decoding). Then load event timestamps from `events.h5` and compute the temporal overlap window.

> **DSEC quirk:** `events.h5` uses BLOSC compression (`import hdf5plugin` required) and stores timestamps as *relative* µs. Absolute timestamp = `events/t` + `t_offset`.

In [23]:
import numpy as np
from rosbags.rosbag1 import Reader

# 2a. Collect LiDAR timestamps (ns → µs)
lidar_timestamps = []

with Reader(BAG_PATH) as reader:
    for connection, timestamp, rawdata in reader.messages():
        if connection.topic == LIDAR_TOPIC:
            lidar_timestamps.append(timestamp // 1000)  # ns → µs

lidar_timestamps = np.array(sorted(lidar_timestamps), dtype=np.int64)

duration_lidar = (lidar_timestamps[-1] - lidar_timestamps[0]) / 1e6
sweep_rate     = len(lidar_timestamps) / duration_lidar

print(f"Total LiDAR sweeps : {len(lidar_timestamps):,}")
print(f"Time range         : {lidar_timestamps[0]:,} µs  →  {lidar_timestamps[-1]:,} µs")
print(f"Duration           : {duration_lidar:.1f} s")
print(f"Mean sweep rate    : {sweep_rate:.1f} Hz")

Total LiDAR sweeps : 6,221
Time range         : 36,338,807,440 µs  →  36,966,156,107 µs
Duration           : 627.3 s
Mean sweep rate    : 9.9 Hz


In [24]:
import hdf5plugin  # must be imported before h5py opens DSEC files (BLOSC compression)
import h5py

# 2b. Inspect events.h5 structure
print("events.h5 key structure:")
with h5py.File(EVENTS_PATH, "r") as f:
    f.visititems(lambda name, obj: print(f"  {name}"))

events.h5 key structure:
  events
  events/p
  events/t
  events/x
  events/y
  ms_to_idx
  t_offset


In [25]:
# 2c. Load event timestamps (DSEC absolute = relative + t_offset)
with h5py.File(EVENTS_PATH, "r") as f:
    t_offset = int(f["t_offset"][()])
    t_rel    = f["events/t"][:]
    t_events = t_rel.astype(np.int64) + t_offset

print(f"t_offset            : {t_offset:,} µs")
print(f"Relative t range    : {t_rel[0]:,}  →  {t_rel[-1]:,} µs")
print(f"Absolute t range    : {t_events[0]:,}  →  {t_events[-1]:,} µs")
print(f"Event count         : {len(t_events):,}")
print(f"Event duration      : {(t_events[-1] - t_events[0]) / 1e6:.1f} s")

t_offset            : 36,470,599,656 µs
Relative t range    : 0  →  35,001,999 µs
Absolute t range    : 36,470,599,656  →  36,505,601,655 µs
Event count         : 358,941,868
Event duration      : 35.0 s


In [27]:
# 2d. Compute overlap window
overlap_start = int(max(t_events[0], lidar_timestamps[0]))
overlap_end   = int(min(t_events[-1], lidar_timestamps[-1]))
overlap_s     = (overlap_end - overlap_start) / 1e6

print(f"Overlap start    : {overlap_start:,} µs")
print(f"Overlap end      : {overlap_end:,} µs")
print(f"Overlap duration : {overlap_s:.1f} seconds")

assert overlap_s > 0, "No temporal overlap between LiDAR and events — check timestamps!"
assert overlap_s > 5, f"Overlap only {overlap_s:.1f}s — suspiciously short, investigate!"

in_overlap = np.sum((lidar_timestamps >= overlap_start) & (lidar_timestamps <= overlap_end))
print(f"\nLiDAR sweeps within overlap window: {in_overlap}")

np.save(BOUNDS_PATH, np.array([overlap_start, overlap_end], dtype=np.int64))
print(f"overlap_bounds_04a.npy saved ")

Overlap start    : 36,470,599,656 µs
Overlap end      : 36,505,601,655 µs
Overlap duration : 35.0 seconds

LiDAR sweeps within overlap window: 347
overlap_bounds_04a.npy saved 



## 3. Extract LiDAR Point Clouds for the Overlap Window

Deserialise only sweeps within the overlap window. Save each as `{timestamp_us:016d}.npy` with shape `(N, 4)` -> columns: x, y, z, intensity.

> **rosbags ≥ 0.9 note:** `deserialize_cdr` was removed. Use `typestore.deserialize_ros1(rawdata, msgtype)` instead.

In [29]:
import struct
from rosbags.typesys import Stores, get_typestore

typestore = get_typestore(Stores.ROS1_NOETIC)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#print(f"Output directory: {OUTPUT_DIR}")


def parse_pointcloud2(msg):
    """Extract x, y, z, intensity from a PointCloud2 message.
    Returns (N, 4) float32 array of valid (non-NaN, non-zero) points.
    """
    field_map        = {f.name: f.offset for f in msg.fields}
    point_step       = msg.point_step
    data             = bytes(msg.data)
    n_points         = len(data) // point_step
    intensity_offset = field_map.get("intensity", field_map.get("i", None))

    pts = np.zeros((n_points, 4), dtype=np.float32)
    for i in range(n_points):
        base      = i * point_step
        pts[i, 0] = struct.unpack_from("f", data, base + field_map.get("x", 0))[0]
        pts[i, 1] = struct.unpack_from("f", data, base + field_map.get("y", 4))[0]
        pts[i, 2] = struct.unpack_from("f", data, base + field_map.get("z", 8))[0]
        if intensity_offset is not None:
            pts[i, 3] = struct.unpack_from("f", data, base + intensity_offset)[0]

    valid = np.isfinite(pts[:, :3]).all(axis=1) & (np.abs(pts[:, :3]).sum(axis=1) > 0)
    return pts[valid]


print("typestore ready. parse_pointcloud2 defined.")

typestore ready. parse_pointcloud2 defined.


In [30]:
# ── Main extraction loop ──────────────────────────────────────────────────────
saved_count   = 0
skipped_count = 0
error_count   = 0

print(f"Extracting sweeps for '{LIDAR_TOPIC}' ...")
print(f"Overlap window: {overlap_start:,} – {overlap_end:,} µs\n")

with Reader(BAG_PATH) as reader:
    for connection, timestamp, rawdata in reader.messages():
        if connection.topic != LIDAR_TOPIC:
            continue

        ts_us = timestamp // 1000

        if not (overlap_start <= ts_us <= overlap_end):
            skipped_count += 1
            continue

        try:
            msg    = typestore.deserialize_ros1(rawdata, connection.msgtype)
            points = parse_pointcloud2(msg)
            np.save(OUTPUT_DIR / f"{ts_us:016d}.npy", points)
            saved_count += 1
            if saved_count % 50 == 0:
                print(f"  ... saved {saved_count} sweeps")
        except Exception as exc:
            error_count += 1
            if error_count <= 3:
                print(f"  [WARN] ts={ts_us}: {exc}")

print(f"\nExtraction complete.")
print(f"  Saved         : {saved_count} sweeps")
print(f"  Skipped (OOW) : {skipped_count} sweeps")
print(f"  Errors        : {error_count}")

Extracting sweeps for '/velodyne_points' ...
Overlap window: 36,470,599,656 – 36,505,601,655 µs

  ... saved 50 sweeps
  ... saved 100 sweeps
  ... saved 150 sweeps
  ... saved 200 sweeps
  ... saved 250 sweeps
  ... saved 300 sweeps

Extraction complete.
  Saved         : 347 sweeps
  Skipped (OOW) : 5874 sweeps
  Errors        : 0


In [31]:
#  Verify a sample sweep
sample_files = sorted(OUTPUT_DIR.glob("*.npy"))
assert len(sample_files) > 0, "No .npy files found in output directory"

mid    = sample_files[len(sample_files) // 2]
sample = np.load(mid)

print(f"Total .npy files : {len(sample_files)}")
print(f"Sample file      : {mid.name}")
print(f"  Shape          : {sample.shape}")
print(f"  X range        : {sample[:, 0].min():.1f}  to  {sample[:, 0].max():.1f} m")
print(f"  Y range        : {sample[:, 1].min():.1f}  to  {sample[:, 1].max():.1f} m")
print(f"  Z range        : {sample[:, 2].min():.1f}  to  {sample[:, 2].max():.1f} m")
print(f"  Intensity      : {sample[:, 3].min():.1f} – {sample[:, 3].max():.1f}")

Total .npy files : 347
Sample file      : 0000036488080138.npy
  Shape          : (24952, 4)
  X range        : -67.8  to  104.6 m
  Y range        : -49.3  to  47.7 m
  Z range        : -2.9  to  14.0 m
  Intensity      : 0.0 – 209.0



## 4. Temporal Matching: Pair Each Semantic Frame with Nearest LiDAR Sweep

In [32]:
import json

frame_timestamps = np.loadtxt(TIMESTAMPS_PATH, dtype=np.int64)

frame_duration = (frame_timestamps[-1] - frame_timestamps[0]) / 1e6
frame_rate     = len(frame_timestamps) / frame_duration

print(f"Total semantic frames   : {len(frame_timestamps)}")
print(f"Frame time range        : {frame_timestamps[0]:,}  →  {frame_timestamps[-1]:,} µs")
print(f"Sample timestamps       : {frame_timestamps[:3]}")
print(f"Approx frame rate       : {frame_rate:.1f} Hz")

Total semantic frames   : 351
Frame time range        : 36,470,600,624  →  36,505,600,853 µs
Sample timestamps       : [36470600624 36470700641 36470800672]
Approx frame rate       : 10.0 Hz


In [33]:
# Match each frame to nearest LiDAR sweep
MAX_DT_MS = 55.0

matched_pairs = []
for i, frame_ts in enumerate(frame_timestamps):
    nearest_idx      = int(np.argmin(np.abs(lidar_timestamps - frame_ts)))
    nearest_lidar_ts = int(lidar_timestamps[nearest_idx])
    dt_ms            = abs(int(frame_ts) - nearest_lidar_ts) / 1000.0
    matched_pairs.append({
        "frame_idx"  : i,
        "frame_ts"   : int(frame_ts),
        "lidar_ts"   : nearest_lidar_ts,
        "dt_ms"      : round(dt_ms, 3),
        "lidar_file" : f"{nearest_lidar_ts:016d}.npy",
    })

valid_pairs = [p for p in matched_pairs if p["dt_ms"] < MAX_DT_MS]

print(f"Total frames            : {len(matched_pairs)}")
print(f"Valid pairs (dt<{MAX_DT_MS}ms) : {len(valid_pairs)}")
print(f"Rejected                : {len(matched_pairs) - len(valid_pairs)}")

if valid_pairs:
    dts = [p["dt_ms"] for p in valid_pairs]
    print(f"Mean temporal gap       : {np.mean(dts):.2f} ms")
    print(f"Max  temporal gap       : {np.max(dts):.2f} ms")
    print(f"Median temporal gap     : {np.median(dts):.2f} ms")

Total frames            : 351
Valid pairs (dt<55.0ms) : 351
Rejected                : 0
Mean temporal gap       : 25.20 ms
Max  temporal gap       : 50.33 ms
Median temporal gap     : 25.06 ms


In [35]:
#  Save pairs JSON
PAIRS_OUTPUT.mkdir(parents=True, exist_ok=True)
pairs_path = PAIRS_OUTPUT / "zurich_city_04_a_pairs.json"

with open(pairs_path, "w") as f:
    json.dump(valid_pairs, f, indent=2)

print(f"Saved {len(valid_pairs)} pairs")
print("\nFirst pair:", json.dumps(valid_pairs[0], indent=2))
print("\nLast pair: ", json.dumps(valid_pairs[-1], indent=2))

Saved 351 pairs

First pair: {
  "frame_idx": 0,
  "frame_ts": 36470600624,
  "lidar_ts": 36470631369,
  "dt_ms": 30.745,
  "lidar_file": "0000036470631369.npy"
}

Last pair:  {
  "frame_idx": 350,
  "frame_ts": 36505600853,
  "lidar_ts": 36505629765,
  "dt_ms": 28.912,
  "lidar_file": "0000036505629765.npy"
}


In [39]:
#Sanity-check: verify every lidar_file exists on disk
missing = [p["lidar_file"] for p in valid_pairs if not (OUTPUT_DIR / p["lidar_file"]).exists()]

if missing:
    # Drop boundary pairs whose sweep fell just outside the overlap window
    valid_pairs = [p for p in valid_pairs if (OUTPUT_DIR / p["lidar_file"]).exists()]
    print(f"Dropped {len(missing)} boundary pair(s) with missing .npy — re-saving JSON.")
    with open(pairs_path, "w") as f:
        json.dump(valid_pairs, f, indent=2)

print(f"\n All {len(valid_pairs)} lidar_file paths verified on disk.")
print(f"\n VALID PAIR COUNT: {len(valid_pairs)}")


 All 350 lidar_file paths verified on disk.

 VALID PAIR COUNT: 350


---
The extraction and alignment pipeline ran successfully on `zurich_city_04_a`.

Key numbers:
- LiDAR sweep rate: 9.9 Hz (one sweep every ~101ms)
- Event camera coverage: 35.0 seconds
- LiDAR sweeps inside the overlap window: 347
- Semantic label frames: 351
- Valid matched pairs after 55ms filter: **350**
- Mean temporal gap between matched pairs: 25.19ms
- Maximum temporal gap: 50.33ms

350 training samples is sufficient for a 3-layer shallow network comparison. Each sample is a triplet: a semantic label frame (ground truth), its nearest LiDAR sweep (Branch B input after projection in Session 2), and the corresponding event window for Time Surface construction (Branch A input).

The one rejected pair was a boundary case where the nearest LiDAR sweep fell just outside the overlap window. No data quality issues were found otherwise.

Output: `data/zurich_city_04_a_pairs.json` (350 entries, committed to repo)